# LLM Asset Classification Verification 

In [54]:
import pandas as pd
import os
from sentence_transformers import SentenceTransformer

data = pd.read_csv('test.csv')
#data = data[:200]
data

,Entity_number,Description,Category,Class,Tag,Valid_Class
0,NX5232A,"Chiller, Silo Bldg Control Room","H.V.A.C.,Chiller",HVAC,NaN,http://www.toronto.ca/TWONTO#chiller
1,TAB-WA1-SQ-1984,"Lanyard 6 Ft- Velasco, Gabriel\t\t\t\t\t\t\t","PPE,Harness",Safety Equipment,SQ,lifeline
2,THC-ACC-HTR-6025,"Heater, Unit, Electric, Heating System, Lower ...","H.V.A.C.,Heater,Unit",HVAC,HTR,http://www.toronto.ca/TWONTO#heater
3,TAB-WA4-SQ-3430,Fall Limiter - MFLT2/705F,"PPE,Lanyard",Safety Equipment,SQ,http://www.toronto.ca/TWONTO#fall_restricting_...
4,THC-ACC-PDIT-6291,"Transmitter, Pressure Differential, Filter F-6...","Transmitter,Pressure",HVAC,PDIT,http://www.toronto.ca/TWONTO#pressure_transmitter
...,...,...,...,...,...,...
24602,NaN,NaN,NaN,NaN,NaN,NaN
24603,NaN,NaN,NaN,NaN,NaN,NaN
24604,NaN,NaN,NaN,NaN,NaN,NaN
24605,NaN,NaN,NaN,NaN,NaN,NaN


In [55]:
import requests
from huggingface_hub import configure_http_backend

def backend_factory() -> requests.Session:
    session = requests.Session()
    session.verify = False
    return session

configure_http_backend(backend_factory=backend_factory)

In [56]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# The sentences to encode
sentences = data["Description"].dropna()  # Remove NaNs
sentences = sentences[sentences.apply(lambda x: isinstance(x, str))]  # Keep only strings
sentences = sentences.tolist()

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])

C:\Users\medney\AppData\Roaming\Python\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\medney\AppData\Roaming\Python\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\medney\AppData\Roaming\Python\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C

(100, 384)
tensor([[ 1.0000,  0.2076,  0.3971,  ...,  0.0663,  0.1156,  0.4155],
        [ 0.2076,  1.0000,  0.0911,  ...,  0.2016,  0.2940,  0.1461],
        [ 0.3971,  0.0911,  1.0000,  ..., -0.0129,  0.0280,  0.5729],
        ...,
        [ 0.0663,  0.2016, -0.0129,  ...,  1.0000,  0.5391,  0.0429],
        [ 0.1156,  0.2940,  0.0280,  ...,  0.5391,  1.0000,  0.0821],
        [ 0.4155,  0.1461,  0.5729,  ...,  0.0429,  0.0821,  1.0000]])


In [61]:
for ridx, row in enumerate(similarities):
    print(sentences[ridx], end=': ')
    for cidx, col in enumerate(row):
        if col > 0.9 and col < 1:
            print(sentences[cidx], end='-')

    print('\n', end='')

Chiller, Silo Bldg Control Room: 
Lanyard  6 Ft- Velasco, Gabriel							: 
Heater, Unit, Electric, Heating System, Lower Level 2, Stair C, Headworks: 
Fall Limiter - MFLT2/705F: 
Transmitter, Pressure Differential, Filter F-6291, 2nd Floor, Mech Room #2, Headworks: 
Switch, Flow, Low, Water Scour, Aerated Grit Tank T-0300, North Grit Building: 
Body Harness Xlarge- Hitish Mistry: Body Harness Xlarge- Hitish Mistry-
Come Along, Work Area 2: Come Along, Work Area 2-
Flow Switch, Low, Cooling Water Supply to Heat Exchanger: 
Sling, Nylon, Work Area 1 (NYLON BELT SLING SIZE; 6" X 13` CAPACITY 16800Lbs ): 
Switch, Pressure-Low, Main Oil Pump, Blower 1911, Secondary Treatment, Aeration Blower System - North: Switch, Pressure-Low, Main Oil Pump, Blower 1911, Secondary Treatment, Aeration Blower System - North-
Power Distribution Panel, 600V, Filter Control Gallery, Inlet/Outlet Sluice Gates: 
Spill Containment Kit Station, TW Building, S2, Pipe Gallery, South of Service Elevator Shaft: Spill 